# WhereAmI: Intelligent Indoor Localization Using Wi-Fi Fingerprints

**PRML Course Project — Complete, corrected and reproducible notebook**

This notebook develops an end-to-end indoor-localization study using the **UJIIndoorLoc** Wi-Fi fingerprint dataset.

## Project objectives
1. Predict the building and floor from Wi-Fi received-signal-strength (RSSI) fingerprints.
2. Estimate indoor coordinates (longitude and latitude in the dataset's local coordinate system).
3. Compare classical pattern-recognition methods: KNN, PCA + KNN, Gaussian Naive Bayes and Random Forest.
4. Study robustness when Wi-Fi access-point measurements are missing or corrupted.
5. Estimate prediction reliability using KNN vote confidence and split-conformal prediction sets.
6. Explore fingerprint structure using K-Means clustering.

> **Important:** This is a dataset-based prototype, not a live Wi-Fi scanner. The dataset coordinates are not a ready-made map. A floor-plan overlay requires a compatible floor plan and coordinate calibration.

## How to run
Run every cell from top to bottom. The dataset cell first downloads the official UCI ZIP archive directly (the previous `ucimlrepo` import path is not reliable for this dataset). If download is unavailable, place `trainingData.csv` and `validationData.csv` inside a `data/` folder beside this notebook.

The official dataset provides a reference/training file and a separate validation/test file. This notebook keeps that separation: model fitting uses `trainingData.csv`, and final evaluation uses `validationData.csv`.


## 0. Install and import dependencies
Run this cell once in the active Python environment.

In [ ]:
%pip -q install scikit-learn pandas numpy matplotlib seaborn scipy joblib

In [ ]:
import os
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.decomposition import PCA
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.cluster import KMeans
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, classification_report,
    confusion_matrix, mean_absolute_error, mean_squared_error,
    median_absolute_error, r2_score, silhouette_score
)
from sklearn.calibration import calibration_curve
from sklearn.inspection import permutation_importance
from joblib import dump

warnings.filterwarnings("ignore")
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 140)

print("Libraries imported successfully.")

## 1. Load the UJIIndoorLoc dataset

The official UJIIndoorLoc dataset contains 19,937 reference/training records and 1,111 separate validation records. Each row contains 520 Wi-Fi access-point (WAP) RSSI measurements and 9 metadata/target fields.

The notebook downloads the official ZIP archive from the UCI repository. If automatic download is blocked, place both CSV files in `./data/` and rerun the loading cell.


In [ ]:
import urllib.request
import zipfile

DATA_DIR = Path("data")
DATA_DIR.mkdir(parents=True, exist_ok=True)

train_path = DATA_DIR / "trainingData.csv"
valid_path = DATA_DIR / "validationData.csv"
zip_path = DATA_DIR / "UJIndoorLoc.zip"

dataset_source = None
train_df = None
validation_df = None

# 1) Prefer already downloaded CSV files.
if train_path.exists() and valid_path.exists():
    train_df = pd.read_csv(train_path)
    validation_df = pd.read_csv(valid_path)
    dataset_source = "Local UJIIndoorLoc CSV files"

else:
    # 2) Download the official UCI archive and extract the two CSV files.
    dataset_url = "https://archive.ics.uci.edu/static/public/310/ujiindoorloc.zip"
    try:
        print("Downloading UJIIndoorLoc from the official UCI repository...")
        request = urllib.request.Request(
            dataset_url,
            headers={"User-Agent": "Mozilla/5.0"}
        )
        with urllib.request.urlopen(request, timeout=60) as response:
            zip_path.write_bytes(response.read())

        with zipfile.ZipFile(zip_path, "r") as archive:
            members = archive.namelist()
            train_member = next(
                (name for name in members if Path(name).name.lower() == "trainingdata.csv"),
                None
            )
            valid_member = next(
                (name for name in members if Path(name).name.lower() == "validationdata.csv"),
                None
            )
            if train_member is None or valid_member is None:
                raise FileNotFoundError(
                    "The downloaded archive does not contain both expected CSV files."
                )

            with archive.open(train_member) as file:
                train_df = pd.read_csv(file)
            with archive.open(valid_member) as file:
                validation_df = pd.read_csv(file)

        # Save extracted CSVs for future runs (no need to download again).
        train_df.to_csv(train_path, index=False)
        validation_df.to_csv(valid_path, index=False)
        dataset_source = "Official UCI ZIP download"

    except Exception as download_error:
        print("Automatic download failed:", repr(download_error))
        if train_path.exists():
            train_df = pd.read_csv(train_path)
            validation_df = pd.read_csv(valid_path) if valid_path.exists() else None
            dataset_source = "Local CSV fallback"
        else:
            raise RuntimeError(
                "Could not download UJIIndoorLoc. Download the dataset from "
                "https://archive.ics.uci.edu/dataset/310/ujiindoorloc, extract "
                "trainingData.csv and validationData.csv, and place both files "
                "inside a folder named 'data' beside this notebook. Then rerun this cell."
            ) from download_error

# Standardize column names once, immediately after loading.
train_df.columns = [str(c).strip().upper() for c in train_df.columns]
if validation_df is not None:
    validation_df.columns = [str(c).strip().upper() for c in validation_df.columns]

print("Dataset source:", dataset_source)
print("Training data shape:", train_df.shape)
print("Validation data shape:", None if validation_df is None else validation_df.shape)
print("Training and validation data loaded successfully.")

## 2. Inspect and understand the data

WAP columns contain RSSI measurements in dBm. A value of `100` means that the access point was not detected; it is a sentinel value, not a real signal strength. We preserve it during initial inspection and convert it to missing values before model training.


In [ ]:
print("Source:", dataset_source)
print("Shape:", train_df.shape)
display(train_df.head())
print("\nColumn names (first 25):", train_df.columns.tolist()[:25])
print("\nData types:")
display(train_df.dtypes.value_counts())
print("\nMissing values (top 15):")
display(train_df.isna().sum().sort_values(ascending=False).head(15))

In [ ]:
# Identify Wi-Fi fingerprint columns and metadata columns.
wap_cols = [c for c in train_df.columns if c.startswith("WAP")]
metadata_candidates = [
    "LONGITUDE", "LATITUDE", "FLOOR", "BUILDINGID", "SPACEID",
    "RELATIVEPOSITION", "USERID", "PHONEID", "TIMESTAMP"
]
metadata_cols = [c for c in metadata_candidates if c in train_df.columns]

print(f"Number of WAP features: {len(wap_cols)}")
print("Metadata columns found:", metadata_cols)

if len(wap_cols) == 0:
    raise ValueError(
        "No WAP columns were found. Check that the correct UJIIndoorLoc CSV files were loaded."
    )

if len(wap_cols) != 520:
    warnings.warn(f"Expected 520 WAP columns, but found {len(wap_cols)}.")

required_columns = {"BUILDINGID", "FLOOR"}
missing_columns = required_columns.difference(train_df.columns)
if missing_columns:
    raise ValueError(f"Training data is missing required columns: {sorted(missing_columns)}")

if validation_df is not None:
    missing_valid = required_columns.difference(validation_df.columns)
    if missing_valid:
        raise ValueError(f"Validation data is missing required columns: {sorted(missing_valid)}")

display(train_df[metadata_cols].describe(include="all").T if metadata_cols else pd.DataFrame())

## 3. Exploratory data analysis

In [ ]:
# Distribution of building and floor labels
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
if "BUILDINGID" in train_df:
    train_df["BUILDINGID"].value_counts().sort_index().plot(kind="bar", ax=axes[0])
    axes[0].set_title("Samples per building")
    axes[0].set_xlabel("Building ID")
if "FLOOR" in train_df:
    train_df["FLOOR"].value_counts().sort_index().plot(kind="bar", ax=axes[1])
    axes[1].set_title("Samples per floor")
    axes[1].set_xlabel("Floor")
plt.tight_layout()
plt.show()

# RSSI distribution: 100 is the missing/not-detected sentinel.
sample_waps = wap_cols[:min(12, len(wap_cols))]
plt.figure(figsize=(12, 5))
for col in sample_waps[:6]:
    values = train_df[col].replace(100, np.nan).dropna()
    if len(values):
        values.plot(kind="kde", label=col)
plt.title("RSSI distributions for selected access points")
plt.xlabel("RSSI value in dBm")
plt.legend()
plt.show()

In [ ]:
# Number of access points detected per sample (RSSI < 100).
detected_count = (train_df[wap_cols] < 100).sum(axis=1)
plt.figure(figsize=(8, 4))
plt.hist(detected_count, bins=30, edgecolor="black")
plt.title("Detected Wi-Fi access points per fingerprint")
plt.xlabel("Number of detected access points")
plt.ylabel("Number of samples")
plt.show()
print(detected_count.describe())

In [ ]:
# Correlation heatmap for a manageable subset of WAP features.
corr_cols = wap_cols[:min(25, len(wap_cols))]
corr_matrix = train_df[corr_cols].corr()
plt.figure(figsize=(12, 9))
image = plt.imshow(corr_matrix, cmap="coolwarm", vmin=-1, vmax=1, aspect="auto")
plt.colorbar(image, label="Correlation")
plt.xticks(range(len(corr_cols)), corr_cols, rotation=90)
plt.yticks(range(len(corr_cols)), corr_cols)
plt.title("Correlation among a subset of Wi-Fi access points")
plt.tight_layout()
plt.show()

## 4. Prepare features and targets

Primary classification target: **building-floor combination**. This is a meaningful hierarchical localization task.

Coordinate regression target: `LONGITUDE` and `LATITUDE`, when available.

Only WAP columns are used as model inputs. Metadata such as building, floor, coordinates, user ID and timestamp are targets or descriptive fields, not input features. This avoids target leakage.

The official validation file is held out for final evaluation. If only a training CSV is available, the notebook creates a reproducible stratified holdout instead.


In [ ]:
# Convert Wi-Fi measurements to numeric values and treat 100 as not detected.
X = train_df[wap_cols].apply(pd.to_numeric, errors="coerce").replace(100, np.nan)

y_building = train_df["BUILDINGID"].astype(str)
y_floor = train_df["FLOOR"].astype(str)
y_building_floor = y_building + "_floor_" + y_floor

has_coordinates = all(c in train_df.columns for c in ["LONGITUDE", "LATITUDE"])
y_coordinates = (
    train_df[["LONGITUDE", "LATITUDE"]].apply(pd.to_numeric, errors="coerce")
    if has_coordinates else None
)

# Prepare the official validation data when it is available.
if validation_df is not None:
    missing_waps = sorted(set(wap_cols) - set(validation_df.columns))
    if missing_waps:
        raise ValueError(
            f"Validation CSV is missing {len(missing_waps)} WAP columns; "
            f"first missing columns: {missing_waps[:10]}"
        )
    X_external = validation_df[wap_cols].apply(
        pd.to_numeric, errors="coerce"
    ).replace(100, np.nan)
    y_external = (
        validation_df["BUILDINGID"].astype(str)
        + "_floor_"
        + validation_df["FLOOR"].astype(str)
    )
    external_has_coordinates = all(
        c in validation_df.columns for c in ["LONGITUDE", "LATITUDE"]
    )
    y_coordinates_external = (
        validation_df[["LONGITUDE", "LATITUDE"]].apply(pd.to_numeric, errors="coerce")
        if external_has_coordinates else None
    )
else:
    X_external = None
    y_external = None
    y_coordinates_external = None
    external_has_coordinates = False

print("Training feature matrix:", X.shape)
print("Building-floor classes:", y_building_floor.nunique())
print("Class counts (most common):")
display(y_building_floor.value_counts().head(15))

In [ ]:
# Use the official held-out validation set whenever it is available.
if X_external is not None and y_external is not None:
    X_train, X_test = X.copy(), X_external.copy()
    y_train, y_test = y_building_floor.copy(), y_external.copy()
    evaluation_protocol = "Official UJIIndoorLoc validation set"
else:
    # Fallback for a training-only local dataset.
    min_class_count = y_building_floor.value_counts().min()
    stratify_target = y_building_floor if min_class_count >= 2 else None
    X_train, X_test, y_train, y_test = train_test_split(
        X, y_building_floor,
        test_size=0.25,
        random_state=RANDOM_STATE,
        stratify=stratify_target
    )
    evaluation_protocol = "Stratified random holdout from training data"

print("Evaluation protocol:", evaluation_protocol)
print("Train:", X_train.shape, "Test:", X_test.shape)
print("Train classes:", y_train.nunique(), "Test classes:", y_test.nunique())
unseen_test_classes = sorted(set(y_test.unique()) - set(y_train.unique()))
print("Classes present only in test:", unseen_test_classes)

## 5. Baseline model: most-frequent class

A baseline is essential: a sophisticated model should outperform a simple reference method.

In [ ]:
from sklearn.dummy import DummyClassifier

baseline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("model", DummyClassifier(strategy="most_frequent"))
])
baseline.fit(X_train, y_train)
baseline_pred = baseline.predict(X_test)
baseline_accuracy = accuracy_score(y_test, baseline_pred)
print(f"Most-frequent baseline accuracy: {baseline_accuracy:.4f}")

## 6. KNN fingerprint classifier

KNN is a natural starting point for fingerprint localization: fingerprints with similar RSSI patterns are expected to correspond to nearby reference locations.

We compare a few values of `k`. Scaling is performed after imputation. The missing-value imputer uses the median RSSI for each access point.

In [ ]:
knn_results = []
knn_models = {}

for k in [1, 3, 5, 7, 9]:
    model = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("knn", KNeighborsClassifier(n_neighbors=k, weights="distance", n_jobs=-1))
    ])
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    score = accuracy_score(y_test, pred)
    knn_results.append({"k": k, "accuracy": score, "balanced_accuracy": balanced_accuracy_score(y_test, pred)})
    knn_models[k] = model

knn_results_df = pd.DataFrame(knn_results)
display(knn_results_df)
best_k = int(knn_results_df.sort_values("accuracy", ascending=False).iloc[0]["k"])
best_knn = knn_models[best_k]
print("Best k:", best_k)

In [ ]:
best_pred = best_knn.predict(X_test)
print(classification_report(y_test, best_pred, zero_division=0))

## 7. PCA + KNN

PCA transforms correlated WAP measurements into a smaller set of orthogonal components. We compare different retained dimensions. The scaler is fitted only on the training data inside each pipeline.

In [ ]:
pca_results = []
pca_models = {}

max_components = min(X_train.shape[0], X_train.shape[1])
candidate_components = [n for n in [10, 20, 40, 60] if n <= max_components]
if not candidate_components:
    candidate_components = [max_components]

for n_components in candidate_components:
    model = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("pca", PCA(n_components=n_components, random_state=RANDOM_STATE)),
        ("knn", KNeighborsClassifier(n_neighbors=best_k, weights="distance", n_jobs=-1))
    ])
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    pca_results.append({
        "components": n_components,
        "accuracy": accuracy_score(y_test, pred),
        "balanced_accuracy": balanced_accuracy_score(y_test, pred)
    })
    pca_models[n_components] = model

pca_results_df = pd.DataFrame(pca_results)
display(pca_results_df)
best_pca_n = int(pca_results_df.sort_values("accuracy", ascending=False).iloc[0]["components"])
best_pca_knn = pca_models[best_pca_n]
print("Best PCA component count:", best_pca_n)

In [ ]:
# Explained variance for a PCA fit on the training features.
imputer_for_pca = SimpleImputer(strategy="median")
scaler_for_pca = StandardScaler()
X_train_imp = imputer_for_pca.fit_transform(X_train)
X_train_scaled = scaler_for_pca.fit_transform(X_train_imp)
pca_full = PCA(random_state=RANDOM_STATE).fit(X_train_scaled)
cum_var = np.cumsum(pca_full.explained_variance_ratio_)

plt.figure(figsize=(8, 4))
plt.plot(np.arange(1, len(cum_var) + 1), cum_var)
plt.axhline(0.90, linestyle="--", label="90% variance")
plt.xlabel("Number of principal components")
plt.ylabel("Cumulative explained variance")
plt.title("PCA explained variance")
plt.legend()
plt.show()

## 8. Probabilistic baseline: Gaussian Naive Bayes

Gaussian Naive Bayes estimates class-conditional feature distributions and applies Bayes' rule. Its conditional-independence assumption is simplified for correlated Wi-Fi signals, which makes it useful as a probabilistic baseline.

In [ ]:
gnb = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("model", GaussianNB())
])
gnb.fit(X_train, y_train)
gnb_pred = gnb.predict(X_test)
print("Gaussian Naive Bayes accuracy:", accuracy_score(y_test, gnb_pred))
print("Gaussian Naive Bayes balanced accuracy:", balanced_accuracy_score(y_test, gnb_pred))

## 9. Random Forest comparison

Random Forest provides a nonlinear tree-ensemble comparison. It is included as a benchmark; the central emphasis remains on understanding classical pattern-recognition methods.

In [ ]:
rf = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("model", RandomForestClassifier(
        n_estimators=250, max_features="sqrt",
        min_samples_leaf=1, class_weight="balanced_subsample",
        random_state=RANDOM_STATE, n_jobs=-1
    ))
])
rf.fit(X_train, y_train)
rf_pred = rf.predict(X_test)
print("Random Forest accuracy:", accuracy_score(y_test, rf_pred))
print("Random Forest balanced accuracy:", balanced_accuracy_score(y_test, rf_pred))

## 10. Consolidated classification comparison

In [ ]:
classification_comparison = pd.DataFrame([
    {"Model": "Most-frequent baseline", "Accuracy": baseline_accuracy,
     "Balanced accuracy": balanced_accuracy_score(y_test, baseline_pred)},
    {"Model": f"KNN (k={best_k})", "Accuracy": accuracy_score(y_test, best_pred),
     "Balanced accuracy": balanced_accuracy_score(y_test, best_pred)},
    {"Model": f"PCA + KNN ({best_pca_n} PCs)", "Accuracy": accuracy_score(y_test, best_pca_knn.predict(X_test)),
     "Balanced accuracy": balanced_accuracy_score(y_test, best_pca_knn.predict(X_test))},
    {"Model": "Gaussian Naive Bayes", "Accuracy": accuracy_score(y_test, gnb_pred),
     "Balanced accuracy": balanced_accuracy_score(y_test, gnb_pred)},
    {"Model": "Random Forest", "Accuracy": accuracy_score(y_test, rf_pred),
     "Balanced accuracy": balanced_accuracy_score(y_test, rf_pred)}
]).sort_values("Accuracy", ascending=False)
display(classification_comparison)

plt.figure(figsize=(10, 5))
plt.barh(classification_comparison["Model"], classification_comparison["Accuracy"])
plt.xlim(0, 1)
plt.xlabel("Accuracy")
plt.title("Building-floor classification performance")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

## 11. Confusion matrix

For many classes, a full confusion matrix can become difficult to read. We display the most frequent test classes to make the main error patterns visible.

In [ ]:
top_classes = y_test.value_counts().head(15).index.tolist()
mask = y_test.isin(top_classes)
cm = confusion_matrix(y_test[mask], best_knn.predict(X_test.loc[mask]), labels=top_classes)
plt.figure(figsize=(12, 9))
image = plt.imshow(cm, cmap="Blues", aspect="auto")
plt.colorbar(image, label="Number of samples")
plt.xticks(range(len(top_classes)), top_classes, rotation=90)
plt.yticks(range(len(top_classes)), top_classes)
plt.xlabel("Predicted class")
plt.ylabel("True class")
plt.title("KNN confusion matrix — 15 most frequent classes")
plt.tight_layout()
plt.show()

## 12. Coordinate regression (optional, when coordinates exist)

The coordinate task predicts the two coordinate fields. We report MAE and RMSE in the dataset's coordinate units. Do not interpret these values as metres unless the dataset coordinate reference and units have been verified.

We use the same train/test indices to keep the comparison fair.

In [ ]:
if X_external is not None and y_coordinates_external is not None:
    # Official validation file: same held-out records as the classification experiment.
    Xc_train = X.copy()
    yc_train = y_coordinates.copy()
    Xc_test = X_external.copy()
    yc_test = y_coordinates_external.copy()
elif has_coordinates:
    # Training-only fallback: make one reproducible coordinate holdout.
    valid_coord = y_coordinates.notna().all(axis=1)
    X_coord = X.loc[valid_coord]
    y_coord = y_coordinates.loc[valid_coord]
    Xc_train, Xc_test, yc_train, yc_test = train_test_split(
        X_coord, y_coord, test_size=0.25, random_state=RANDOM_STATE
    )
else:
    Xc_train = Xc_test = yc_train = yc_test = None

if Xc_train is not None:
    train_coord_valid = yc_train.notna().all(axis=1)
    test_coord_valid = yc_test.notna().all(axis=1)
    Xc_train, yc_train = Xc_train.loc[train_coord_valid], yc_train.loc[train_coord_valid]
    Xc_test, yc_test = Xc_test.loc[test_coord_valid], yc_test.loc[test_coord_valid]

    regressors = {
        "KNN regressor": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("model", KNeighborsRegressor(n_neighbors=min(best_k, len(Xc_train)),
                                          weights="distance", n_jobs=-1))
        ]),
        "Random Forest regressor": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("model", RandomForestRegressor(n_estimators=250, min_samples_leaf=1,
                                            random_state=RANDOM_STATE, n_jobs=-1))
        ])
    }

    regression_rows = []
    fitted_regressors = {}
    for name, model in regressors.items():
        model.fit(Xc_train, yc_train)
        pred = model.predict(Xc_test)
        mae = mean_absolute_error(yc_test, pred)
        rmse = np.sqrt(mean_squared_error(yc_test, pred))
        regression_rows.append({
            "Model": name, "MAE": mae, "RMSE": rmse, "R2": r2_score(yc_test, pred)
        })
        fitted_regressors[name] = model

    regression_comparison = pd.DataFrame(regression_rows).sort_values("RMSE")
    display(regression_comparison)
else:
    print("Coordinate columns are unavailable; skipping coordinate regression.")

In [ ]:
if Xc_train is not None and len(yc_test) > 0:
    chosen_regressor = fitted_regressors[regression_comparison.iloc[0]["Model"]]
    coord_pred = chosen_regressor.predict(Xc_test)
    plt.figure(figsize=(7, 7))
    plt.scatter(yc_test["LONGITUDE"], yc_test["LATITUDE"],
                s=12, alpha=0.5, label="Actual")
    plt.scatter(coord_pred[:, 0], coord_pred[:, 1],
                s=12, alpha=0.5, label="Predicted")
    plt.xlabel("Longitude field (dataset coordinates)")
    plt.ylabel("Latitude field (dataset coordinates)")
    plt.title("Actual and predicted coordinate locations")
    plt.legend()
    plt.axis("equal")
    plt.show()
else:
    print("Coordinate comparison is unavailable.")

## 13. Uncertainty and reliability: probability confidence

For KNN, `predict_proba` gives class vote proportions (with distance weighting in this notebook). These values are useful as a confidence proxy, but they are **not automatically calibrated probabilities**.

We inspect whether low maximum confidence tends to coincide with incorrect predictions.

In [ ]:
knn_prob = best_knn.predict_proba(X_test)
knn_classes = best_knn.named_steps["knn"].classes_
max_confidence = knn_prob.max(axis=1)
prob_pred = knn_classes[np.argmax(knn_prob, axis=1)]
is_correct = (prob_pred == y_test.to_numpy())

confidence_df = pd.DataFrame({
    "confidence": max_confidence,
    "correct": is_correct
}, index=y_test.index)

confidence_summary = confidence_df.groupby(
    pd.cut(confidence_df["confidence"], bins=5, duplicates="drop")
)["correct"].agg(["mean", "count"])
display(confidence_summary)

plt.figure(figsize=(7, 4))
plt.hist(confidence_df.loc[confidence_df["correct"], "confidence"],
         bins=20, alpha=0.6, label="Correct")
plt.hist(confidence_df.loc[~confidence_df["correct"], "confidence"],
         bins=20, alpha=0.6, label="Incorrect")
plt.xlabel("Maximum KNN class vote")
plt.ylabel("Number of samples")
plt.title("KNN confidence for correct vs incorrect predictions")
plt.legend()
plt.show()

print("Overall accuracy:", is_correct.mean())
print("Mean maximum class vote:", max_confidence.mean())

## 14. Split-conformal prediction sets

Conformal prediction can provide a set of plausible classes rather than forcing a single answer. We split the training partition into a model-fit subset and a calibration subset, compute nonconformity scores `1 - p(true class)`, and select a quantile for the requested coverage level.

The guarantee relies on exchangeability between calibration and future test examples. It does not guarantee that the true class is included under arbitrary distribution shift.

In [ ]:
# Split the training partition into model-fit and calibration subsets.
# Stratification is used only when every class has at least two observations.
calibration_stratify = y_train if y_train.value_counts().min() >= 2 else None
X_fit, X_cal, y_fit, y_cal = train_test_split(
    X_train, y_train, test_size=0.20,
    random_state=RANDOM_STATE, stratify=calibration_stratify
)

conformal_model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("knn", KNeighborsClassifier(
        n_neighbors=min(best_k, len(X_fit)), weights="distance", n_jobs=-1
    ))
])
conformal_model.fit(X_fit, y_fit)

cal_probs = conformal_model.predict_proba(X_cal)
cal_classes = conformal_model.named_steps["knn"].classes_
class_to_col = {label: i for i, label in enumerate(cal_classes)}

# If a rare class is absent from the model-fit subset, its estimated probability is 0.
true_cal_probs = np.array([
    cal_probs[i, class_to_col[label]] if label in class_to_col else 0.0
    for i, label in enumerate(y_cal)
])
nonconformity = 1 - true_cal_probs

alpha = 0.10  # target miscoverage 10%, nominal coverage 90%
n_cal = len(nonconformity)
if n_cal == 0:
    raise ValueError("The calibration set is empty; increase the training dataset size.")
q_level = min(1.0, np.ceil((n_cal + 1) * (1 - alpha)) / n_cal)
q_hat = np.quantile(nonconformity, q_level, method="higher")
print(f"Conformal threshold q_hat: {q_hat:.4f}")

In [ ]:
test_probs_conf = conformal_model.predict_proba(X_test)
test_classes_conf = conformal_model.named_steps["knn"].classes_
prediction_sets = []
for row in test_probs_conf:
    included = test_classes_conf[(1 - row) <= q_hat]
    prediction_sets.append(list(included))

true_labels = y_test.to_numpy()
covered = np.array([true_labels[i] in prediction_sets[i] for i in range(len(true_labels))])
set_sizes = np.array([len(s) for s in prediction_sets])

print(f"Empirical test coverage: {covered.mean():.4f}")
print(f"Mean prediction-set size: {set_sizes.mean():.2f}")
print(f"Median prediction-set size: {np.median(set_sizes):.1f}")

plt.figure(figsize=(8, 4))
plt.hist(set_sizes, bins=np.arange(0, set_sizes.max() + 2) - 0.5, edgecolor="black")
plt.xticks(range(0, int(set_sizes.max()) + 1))
plt.xlabel("Number of candidate classes")
plt.ylabel("Number of samples")
plt.title("Conformal prediction-set size")
plt.show()

## 15. Robustness experiment A: randomly missing access-point readings

We randomly hide a fraction of observed RSSI measurements and evaluate how localization accuracy changes. The imputer and model remain fitted on the original training data. This simulates access points that are intermittently unavailable.

This is a controlled synthetic perturbation, not a claim about a specific real-world Wi-Fi failure rate.

In [ ]:
def mask_observed_measurements(frame, missing_fraction, random_state=RANDOM_STATE):
    rng = np.random.default_rng(random_state)
    arr = frame.to_numpy(dtype=float).copy()
    observed = np.isfinite(arr)
    # Mask only measurements that were observed in the original fingerprint.
    mask = observed & (rng.random(arr.shape) < missing_fraction)
    arr[mask] = np.nan
    return pd.DataFrame(arr, index=frame.index, columns=frame.columns)

robustness_rows = []
for frac in [0.0, 0.10, 0.20, 0.30, 0.40, 0.50]:
    X_perturbed = mask_observed_measurements(X_test, frac, random_state=RANDOM_STATE)
    pred = best_knn.predict(X_perturbed)
    robustness_rows.append({
        "Artificially masked fraction": frac,
        "Accuracy": accuracy_score(y_test, pred),
        "Balanced accuracy": balanced_accuracy_score(y_test, pred)
    })

robustness_df = pd.DataFrame(robustness_rows)
display(robustness_df)
plt.figure(figsize=(8, 4))
plt.plot(robustness_df["Artificially masked fraction"],
         robustness_df["Accuracy"], marker="o")
plt.ylim(0, 1)
plt.xlabel("Artificially masked fraction")
plt.ylabel("Accuracy")
plt.title("Robustness to missing Wi-Fi measurements")
plt.grid(True, alpha=0.3)
plt.show()

## 16. Robustness experiment B: RSSI measurement noise

Add zero-mean Gaussian noise to observed RSSI values. We evaluate performance over increasing noise standard deviations.

In [ ]:
def add_rssi_noise(frame, noise_std, random_state=RANDOM_STATE):
    rng = np.random.default_rng(random_state)
    arr = frame.to_numpy(dtype=float).copy()
    observed = np.isfinite(arr)
    arr[observed] += rng.normal(0, noise_std, size=observed.sum())
    # Keep perturbed RSSI values within the UJIIndoorLoc dataset's documented range.
    arr[observed] = np.clip(arr[observed], -104, 0)
    return pd.DataFrame(arr, index=frame.index, columns=frame.columns)

noise_rows = []
for noise_std in [0, 2, 5, 8, 12]:
    X_noisy = add_rssi_noise(X_test, noise_std, random_state=RANDOM_STATE)
    pred = best_knn.predict(X_noisy)
    noise_rows.append({
        "Noise standard deviation (dB)": noise_std,
        "Accuracy": accuracy_score(y_test, pred),
        "Balanced accuracy": balanced_accuracy_score(y_test, pred)
    })

noise_df = pd.DataFrame(noise_rows)
display(noise_df)
plt.figure(figsize=(8, 4))
plt.plot(noise_df["Noise standard deviation (dB)"],
         noise_df["Accuracy"], marker="o")
plt.ylim(0, 1)
plt.xlabel("Noise standard deviation (dB)")
plt.ylabel("Accuracy")
plt.title("Robustness to RSSI measurement noise")
plt.grid(True, alpha=0.3)
plt.show()

## 17. Unsupervised learning: K-Means on PCA features

Clustering is used here to explore structure in the fingerprints. Cluster IDs do not inherently correspond to building-floor labels. We use silhouette score as an internal metric and compare clusters with known labels only as a descriptive analysis.

For runtime, clustering uses a reproducible sample.

In [ ]:
# Reuse the fitted training-only imputer/scaler and PCA transform.
X_all_imp = imputer_for_pca.transform(X)
X_all_scaled = scaler_for_pca.transform(X_all_imp)
X_pca = pca_full.transform(X_all_scaled)

rng = np.random.default_rng(RANDOM_STATE)
sample_size = min(5000, X_pca.shape[0])
sample_idx = rng.choice(X_pca.shape[0], size=sample_size, replace=False)
X_cluster_sample = X_pca[sample_idx, :min(20, X_pca.shape[1])]

cluster_rows = []
kmeans_models = {}
for n_clusters in [3, 5, 8, 10]:
    km = KMeans(n_clusters=n_clusters, n_init=10, random_state=RANDOM_STATE)
    labels = km.fit_predict(X_cluster_sample)
    sil = silhouette_score(X_cluster_sample, labels) if len(np.unique(labels)) > 1 else np.nan
    cluster_rows.append({"Clusters": n_clusters, "Silhouette score": sil, "Inertia": km.inertia_})
    kmeans_models[n_clusters] = km

cluster_results = pd.DataFrame(cluster_rows)
display(cluster_results)

In [ ]:
best_cluster_count = int(cluster_results.sort_values("Silhouette score", ascending=False).iloc[0]["Clusters"])
best_kmeans = kmeans_models[best_cluster_count]
cluster_labels = best_kmeans.predict(X_cluster_sample)

plt.figure(figsize=(8, 6))
plt.scatter(X_cluster_sample[:, 0], X_cluster_sample[:, 1], c=cluster_labels, s=8, alpha=0.6)
plt.xlabel("Principal component 1")
plt.ylabel("Principal component 2")
plt.title(f"K-Means clusters projected onto first two PCs (k={best_cluster_count})")
plt.show()

## 18. Inspect a single prediction

This cell shows how the model handles one fingerprint. The class vote distribution is shown for the most likely candidate classes.

In [ ]:
sample_position = 0
sample_X = X_test.iloc[[sample_position]]
sample_true = y_test.iloc[sample_position]
sample_prob = best_knn.predict_proba(sample_X)[0]
top_order = np.argsort(sample_prob)[::-1][:5]

print("True building-floor:", sample_true)
print("Predicted building-floor:", best_knn.predict(sample_X)[0])
print("Top candidate classes:")
for idx in top_order:
    print(f"  {best_knn.named_steps['knn'].classes_[idx]}: {sample_prob[idx]:.4f}")

## 19. Save trained models and experiment results

The saved pipeline includes preprocessing steps, so new fingerprints must be supplied with the same WAP columns in the same order. Keep the model and feature-name file together.

In [ ]:
output_dir = Path("whereami_outputs")
output_dir.mkdir(exist_ok=True)

dump(best_knn, output_dir / "whereami_knn_pipeline.joblib")
dump(best_pca_knn, output_dir / "whereami_pca_knn_pipeline.joblib")
dump(rf, output_dir / "whereami_random_forest_pipeline.joblib")

pd.Series(wap_cols, name="feature").to_csv(output_dir / "wap_feature_order.csv", index=False)
classification_comparison.to_csv(output_dir / "classification_comparison.csv", index=False)
if has_coordinates:
    regression_comparison.to_csv(output_dir / "regression_comparison.csv", index=False)
robustness_df.to_csv(output_dir / "missing_measurement_robustness.csv", index=False)
noise_df.to_csv(output_dir / "rssi_noise_robustness.csv", index=False)

print("Saved files in:", output_dir.resolve())
print("Files:", [p.name for p in output_dir.iterdir()])

## 20. Final results summary

Run this section after all experiments. Fill the interpretation using your actual outputs; do not copy generic conclusions without checking the measured values.

In [ ]:
print("========== WHEREAMI RESULTS SUMMARY ==========")
print(f"Dataset rows: {len(train_df):,}")
print(f"Number of WAP features: {len(wap_cols)}")
print(f"Building-floor classes: {y_building_floor.nunique()}")
print(f"Most-frequent baseline accuracy: {baseline_accuracy:.4f}")
print(f"Best KNN (k={best_k}) accuracy: {accuracy_score(y_test, best_pred):.4f}")
print(f"Best PCA+KNN ({best_pca_n} PCs) accuracy: {accuracy_score(y_test, best_pca_knn.predict(X_test)):.4f}")
print(f"GaussianNB accuracy: {accuracy_score(y_test, gnb_pred):.4f}")
print(f"Random Forest accuracy: {accuracy_score(y_test, rf_pred):.4f}")
print(f"Conformal empirical coverage: {covered.mean():.4f}")
print(f"Mean conformal set size: {set_sizes.mean():.2f}")
if "regression_comparison" in globals():
    print("\nCoordinate regression:")
    display(regression_comparison)
print("\nClassification comparison:")
display(classification_comparison)
print("\nMissing measurement robustness:")
display(robustness_df)
print("\nNoise robustness:")
display(noise_df)

## 21. Research discussion and report checklist

Use the actual values generated above to answer these questions:

1. Which classifier achieved the best accuracy and balanced accuracy? Are the two rankings different?
2. Did PCA improve performance, reduce performance, or provide a useful speed/complexity trade-off?
3. How does Gaussian Naive Bayes compare with KNN? What assumptions may explain the difference?
4. How does performance change as more Wi-Fi readings are masked?
5. How does RSSI noise affect the model?
6. What fraction of test examples are covered by conformal prediction sets? What is their average size?
7. Does low KNN confidence correspond to more mistakes? Remember that vote confidence is not automatically calibrated.
8. What are the limitations of a random train/test split? Consider user/device-specific and time-based generalization.

### Important limitations
- The notebook uses a dataset-based evaluation; it does not yet collect live Wi-Fi data.
- The train/test split is random and may not measure generalization to unseen phones, users, buildings or collection sessions.
- RSSI values depend on hardware, orientation, walls, people and environmental conditions.
- Building-floor classification is a coarse localization task. Coordinate regression provides a more fine-grained complementary task.
- Conformal coverage depends on exchangeability; distribution shifts can invalidate nominal guarantees.
- A floor-plan visualization requires a compatible floor plan and coordinate transformation.

### Suggested next research extension
After this baseline notebook runs successfully, we can add a stronger evaluation protocol: hold out one or more users/phones during training and test on those unseen devices. This is a more demanding test of whether the method generalizes beyond the fingerprints it has already seen.

**End of notebook — WhereAmI PRML project.**